# Pass 2: testing the LightGBM ranker as it's built

Scratch notebook for trying out Pass 2 pieces as they land, starting with the interface refactor (`rank_fn`/`optimiser_fn`) and the LightGBM training panel builder. This notebook will grow as `rank_by_lgbm` (the actual fit/predict step) and later Pass 2 items get built — it's not part of the pipeline itself, just a place to poke at things.

Kernel: select **portfolio-constructor** (the project's `.venv`).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data import ingest, features
from src.models import ranker, lgbm_ranker
from src.portfolio import backtest

## 1. Pull data (cached after the first run)

In [ ]:
prices_long = ingest.load_or_fetch_prices(
    config.TICKER_UNIVERSE + [config.BENCHMARK_TICKER], config.DATA_START_DATE, config.DATA_END_DATE
)
dividends = ingest.load_or_fetch_dividends(config.TICKER_UNIVERSE, config.DATA_START_DATE, config.DATA_END_DATE)
prices = ingest.to_wide_adj_close(prices_long)
universe_prices = prices[config.TICKER_UNIVERSE]
print("prices (wide):", prices.shape)

## 2. The refactored `rank_fn` / `optimiser_fn` interface

`backtest.py`'s pluggable ranking rule used to be `score_fn(momentum, low_volatility, dividend_yield)` — three already-computed factor snapshots in, scores out. That doesn't work for a model that needs to build its own training panel across many dates (like LightGBM), so it broadened to `rank_fn(universe_prices, dividends, as_of_date)` — raw data in, scores out. `ranker.rank_by_composite_score` and `ranker.rank_by_momentum_only` are thin adapters that compute the three factors internally and delegate to the original `score_stocks`/`score_by_momentum_only`, so nothing about the composite/momentum-only strategies actually changed — just how they plug in.

In [ ]:
as_of_date = pd.Timestamp("2023-01-03")

composite_scores = ranker.rank_by_composite_score(universe_prices, dividends, as_of_date)
momentum_scores = ranker.rank_by_momentum_only(universe_prices, dividends, as_of_date)

print("Top 5, composite score:")
print(composite_scores.sort_values(ascending=False).head())
print("\nTop 5, momentum-only:")
print(momentum_scores.sort_values(ascending=False).head())

In [ ]:
# Confirm run_backtest still works through the new interface and reproduces
# the same numbers already seen in notebook 1 (default rank_fn/optimiser_fn
# match the old score_fn defaults exactly).
daily_returns, _ = backtest.run_backtest(prices, dividends)
print("composite strategy_net final growth:", (1 + daily_returns["strategy_net"]).cumprod().iloc[-1])

## 3. `features.training_window`'s `extra_months`

The LightGBM panel needs a *wider* window than a single as-of-date query: its earliest training snapshot needs its own 12-month momentum lookback before the window even starts. `extra_months` widens the window's start without moving its end.

In [ ]:
standard_window = features.training_window(universe_prices, as_of_date)
widened_window = features.training_window(universe_prices, as_of_date, extra_months=config.MOMENTUM_LOOKBACK_MONTHS)

print("standard window:", standard_window.index.min().date(), "to", standard_window.index.max().date())
print("widened window: ", widened_window.index.min().date(), "to", widened_window.index.max().date())
print("same end date:", standard_window.index.max() == widened_window.index.max())

## 4. The LightGBM training panel

One row per ticker per historical monthly snapshot: the three factors as of that snapshot date, plus the *realized* forward 1-month return (the label). This is what `rank_by_lgbm` will fit a model on — built here directly to inspect it before that step exists.

In [ ]:
panel = lgbm_ranker._build_training_panel(universe_prices, dividends, as_of_date)

print("panel shape:", panel.shape)
print("snapshot dates:", panel["snapshot_date"].nunique(), "-", panel["snapshot_date"].min().date(), "to", panel["snapshot_date"].max().date())
print("tickers:", panel["ticker"].nunique())
panel.head()

In [ ]:
panel["forward_return"].plot.hist(bins=40, figsize=(9, 4), title="Distribution of forward_return (the label)")
plt.xlabel("forward 1-month return")
plt.show()
panel["forward_return"].describe()

In [ ]:
# Is there any visible relationship between a feature and what actually happened
# next? Weak/noisy is expected and normal for a single factor at the individual
# stock-month level — this is just an eyeball check, not a rigorous signal test.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feature_name in zip(axes, ["momentum", "low_volatility", "dividend_yield"]):
    ax.scatter(panel[feature_name], panel["forward_return"], alpha=0.3, s=10)
    ax.set_xlabel(feature_name)
    ax.set_ylabel("forward_return")
    correlation = panel[feature_name].corr(panel["forward_return"])
    ax.set_title(f"corr = {correlation:.3f}")
plt.tight_layout()
plt.show()

## Next: `rank_by_lgbm`

Not built yet — this section will be extended once the actual `LGBMRegressor` fit/predict function exists, with a full walk-forward comparison against the composite and momentum-only strategies (same pattern as notebook 1's section 6 and notebook 2's Sortino experiment).